# CR Impact Analyzer
## Build Vector Database

### Configuration

In [ ]:
import os

BASE_DIR         = os.getcwd()
PDF_DIR          = os.path.join(BASE_DIR, "pdfs")
VECTOR_STORE_DIR = os.path.join(BASE_DIR, "vector_store")
COLLECTION_NAME  = "system_docs"

CHUNK_SIZE       = 800    # was 500 — bigger chunks keep procedure steps together
CHUNK_OVERLAP    = 150    # was 200 — less redundancy
EMBEDDING_MODEL  = "all-mpnet-base-v2"  # was all-MiniLM-L6-v2 — stronger for technical text
EMBED_BATCH_SIZE = 64

print(" Config ready")
print(f"   Embedding model : {EMBEDDING_MODEL}")
print(f"   Chunk size      : {CHUNK_SIZE} chars")
print(f"   Chunk overlap   : {CHUNK_OVERLAP} chars")

###  Step 1  Extract text from PDFs

In [ ]:
import pymupdf as fitz
import re
import time

def extract_cr_number(filename):
    """Extract CR number from filename e.g. CR-2024-210, PM-2024-115"""
    match = re.search(r'(CR|PM)-?\d{4}-?\d+', filename, re.IGNORECASE)
    return match.group(0).upper() if match else "UNKNOWN"


def clean_text(text):
    """Remove noise from extracted PDF text."""
    text = re.sub(r'\n{3,}', '\n\n', text)   # collapse 3+ newlines to 2
    text = re.sub(r' {3,}', ' ', text)         # collapse multiple spaces
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text) # remove standalone page numbers
    return text.strip()


def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages_text = []
    for page_num, page in enumerate(doc, start=1):
        text = page.get_text("text")
        if text.strip():
            pages_text.append(f"[Page {page_num}]\n{text}")
    doc.close()
    return clean_text("\n\n".join(pages_text))


def extract_all_pdfs(pdf_dir):
    pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]
    if not pdf_files:
        raise ValueError(f"No PDF files found in: {pdf_dir}")
    docs = []
    for fname in sorted(pdf_files):
        path = os.path.join(pdf_dir, fname)
        print(f"  Reading: {fname}")
        text = extract_text_from_pdf(path)
        if text.strip():
            docs.append({
                "source":    fname,
                "cr_number": extract_cr_number(fname),
                "text":      text
            })
        else:
            print(f" Skipped (no extractable text): {fname}")
    return docs


print("Extracting text from PDFs...\n")
t0 = time.time()
docs = extract_all_pdfs(PDF_DIR)
print(f"\n Extracted text from {len(docs)} PDF(s)  ({time.time()-t0:.1f}s)")

###  Step 2 — Chunk the documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

def chunk_document(doc):
    raw_chunks = splitter.split_text(doc["text"])
    return [
        {
            "source":      doc["source"],
            "cr_number":   doc["cr_number"],
            "chunk_index": i,
            "text":        chunk
        }
        for i, chunk in enumerate(raw_chunks)
    ]


print("Chunking documents...\n")
t1 = time.time()
all_chunks = []
for doc in docs:
    chunks = chunk_document(doc)
    print(f"   {doc['source'][:65]}  →  {len(chunks)} chunks")
    all_chunks.extend(chunks)

print(f"\n Total chunks: {len(all_chunks)}  ({time.time()-t1:.1f}s)")

### Step 3 — Generate embeddings & save to ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
print("   (first run downloads ~420 MB — subsequent runs are instant)")
embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(" Embedding model loaded\n")

os.makedirs(VECTOR_STORE_DIR, exist_ok=True)
client = chromadb.PersistentClient(path=VECTOR_STORE_DIR)

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"  Dropped existing collection '{COLLECTION_NAME}'")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

total = len(all_chunks)
print(f"Embedding {total} chunks in batches of {EMBED_BATCH_SIZE} ...\n")
t2 = time.time()

for start in range(0, total, EMBED_BATCH_SIZE):
    batch     = all_chunks[start : start + EMBED_BATCH_SIZE]
    texts     = [c["text"] for c in batch]
    metadatas = [
        {
            "source":      c["source"],
            "cr_number":   c["cr_number"],
            "chunk_index": c["chunk_index"]
        }
        for c in batch
    ]
    ids        = [f"chunk_{start + i}" for i in range(len(batch))]
    embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()

    collection.add(
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
        ids=ids,
    )
    done = min(start + EMBED_BATCH_SIZE, total)
    print(f"   ↳  {done}/{total} chunks embedded")

elapsed = time.time() - t2
print(f"\n Vector store saved!  ({elapsed:.1f}s)")
print(f"   Collection : {COLLECTION_NAME}")
print(f"   Documents  : {collection.count()}")
print(f"   Saved to   : {os.path.abspath(VECTOR_STORE_DIR)}")
print("\n Proceed to Notebook 3!")

###  Optional — Verify a specific CR is indexed

In [ ]:
# Change to any CR number you want to verify
TEST_CR = "CR_To_Test"

results = collection.get(
    where={"cr_number": TEST_CR},
    include=["metadatas", "documents"]
)
chunks_found = len(results["ids"])
print(f"CR {TEST_CR}: {chunks_found} chunks stored")
if chunks_found > 0:
    print(f"\nFirst chunk preview:")
    print("-" * 60)
    print(results["documents"][0][:400])